# Imputation of VIIRS Band Zarr Arrays

## Objective
This notebook performs missing-value imputation for the VIIRS band Zarr archive. The procedure focuses on the `mean`, `min`, and `max` arrays for each band and preserves the auxiliary coordinate and metadata arrays required for downstream feature construction.

## Imputation Strategy
Each target array is processed day by day using a spatial imputation sequence that combines local kernel filling, raster-based interpolation when available, nearest-neighbor propagation, and a final guarantee of finite output.

## Outputs
The notebook writes:
- imputed VIIRS Zarr stores
- per-band summary statistics
- per-day diagnostic statistics

These outputs serve as the cleaned VIIRS input source for the historical-window feature construction pipeline.


## Environment and Imports

This section loads the VIIRS imputation utilities and prepares the local project path so that the imputation module can be imported consistently.


In [1]:
# Optional installs (run only if missing)
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from zarr_feature_imputer_viirs import ViirsImputeConfig, run_viirs_imputation

## Configuration

This section defines the source VIIRS archive, the output directories, the arrays selected for imputation, and the spatial interpolation settings used during processing.


In [2]:
# -------------------------
# Config
# -------------------------
cfg = ViirsImputeConfig(
    source_dir=Path("/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/bands/viirs_all"),
    output_dir=Path("/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs"),
    stats_dir=Path("/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs_stats"),

    # Set to None for all bands in source_dir
    include_bands=None,
    exclude_bands=(),

    # Mean/Min/Max only
    arrays_to_impute=("mean", "min", "max"),

    # Imputation behavior
    kernel_size=3,
    max_passes=1,
    use_rasterio_fillnodata=True,
    rasterio_max_search_distance=100.0,
    rasterio_smoothing_iterations=0,
    nearest_fill_fallback=True,
    ensure_no_nan=True,

    # Parallel across bands
    n_jobs=6,
    show_progress=True,
)

cfg

ViirsImputeConfig(source_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/bands/viirs_all'), output_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs'), stats_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs_stats'), include_bands=None, exclude_bands=(), arrays_to_impute=('mean', 'min', 'max'), preserve_aux_arrays=True, kernel_size=3, max_passes=1, use_rasterio_fillnodata=True, rasterio_max_search_distance=100.0, rasterio_smoothing_iterations=0, nearest_fill_fallback=True, ensure_no_nan=True, n_jobs=6, show_progress=True)

## Imputation Run

The configured VIIRS imputation pipeline is executed here. The returned output object records the paths of the generated Zarr archive and the diagnostic summary files.


In [3]:
# -------------------------
# Run
# -------------------------
out = run_viirs_imputation(cfg)
print(json.dumps(out, indent=2))

bands(parallel):   0%|          | 0/6 [00:00<?, ?band/s]

















































































































































































































































































































































































































































































































































































































































































































































































































































































































































































{
  "summary_csv": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs_stats/viirs_impute_summary.csv",
  "summary_json": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs_stats/viirs_impute_summary.json",
  "day_stats_csv": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs_stats/viirs_impute_day_stats.csv",
  "run_metadata": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs_stats/viirs_impute_run_metadata.json",
  "output_dir": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs"
}


## Diagnostic Inspection

This section provides a compact inspection of the per-band summary table and the per-day diagnostic table. These views allow a direct assessment of the amount of missingness corrected during the VIIRS imputation stage.


In [ ]:
# -------------------------
# Quick summary preview
# -------------------------
import pandas as pd

summary_df = pd.read_csv(out["summary_csv"])
day_df = pd.read_csv(out["day_stats_csv"])

display(summary_df.head(20))
display(day_df.head(20))
print("rows(summary):", len(summary_df))
print("rows(day_stats):", len(day_df))